# The Fairness Gap Is a Map
### A biased fraud model tells an attacker where you aren't looking
**OWASP Boston — August 12, 2026** · Mardiros Merdinian

---
Card-not-present fraud scoring. The model never sees a protected attribute.
It sees payment instrument and account tenure — legitimate risk signals.

**The defect is not in the features. It is in the labels.**

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

SEED = 1337; rng = np.random.default_rng(SEED); N = 60_000

# ---- population: instrument + tenure are LEGITIMATE risk signals ----------
established = rng.binomial(1, 0.62, N)
tenure_days = np.where(established == 1, rng.gamma(9, 130, N),
                                          rng.gamma(1.6, 55, N)).clip(1, 4000)
instrument  = np.where(established == 1, rng.choice([1, 2], N, p=[.30, .70]),
                                          rng.choice([0, 1], N, p=[.68, .32]))
amount          = np.exp(rng.normal(3.9, 1.0, N)).clip(1, 5000)
hour            = rng.integers(0, 24, N)
bill_ship_match = rng.binomial(1, .86, N)
device_reuse    = rng.poisson(1.1, N)
velocity_1h     = rng.poisson(.55, N)
email_age_days  = np.where(established == 1, rng.gamma(3.0, 300, N),
                           rng.gamma(1.4, 110, N)).clip(0, 4000)

# ---- TRUE fraud: driven by behaviour, not by what card you hold -----------
z = (-3.35 + 1.35*(bill_ship_match == 0) + 0.95*(velocity_1h >= 2)
     + 0.80*(device_reuse >= 3) + 0.90*(np.log(amount) > 5.6)
     + 0.75*(email_age_days < 60) + 0.35*((hour >= 1) & (hour <= 5))
     - 0.22*established + rng.normal(0, 0.30, N))
true_fraud = rng.binomial(1, 1/(1+np.exp(-z)))

FULL = ["amount","hour","tenure_days","instrument","bill_ship_match",
        "device_reuse","velocity_1h","email_age_days"]
SAN  = ["amount","hour","bill_ship_match","device_reuse","velocity_1h","email_age_days"]

### The defect

In [2]:
# THE DEFECT: labels come from INVESTIGATIONS, not from reality.
# The legacy rules engine looked at thin-file 74% of the time, established 16%.
p_investigated = np.where(established == 1, 0.16, 0.74)
observed_label = true_fraud * rng.binomial(1, p_investigated)

df = pd.DataFrame(dict(amount=amount, hour=hour, tenure_days=tenure_days,
    instrument=instrument, bill_ship_match=bill_ship_match, device_reuse=device_reuse,
    velocity_1h=velocity_1h, email_age_days=email_age_days,
    established=established, true_fraud=true_fraud, observed_label=observed_label))
sp = rng.random(N) < 0.7; train, test = df[sp].copy(), df[~sp].copy()

def fit(f, label, feats=FULL, w=None):
    m = HistGradientBoostingClassifier(max_iter=180, learning_rate=.08,
                                       max_depth=6, random_state=SEED)
    m.fit(f[feats], f[label], sample_weight=w); return m

model = fit(train, "observed_label")
test["score"] = model.predict_proba(test[FULL])[:, 1]
T0 = np.quantile(test.score, .93)          # ~7% decline budget
test["d0"] = (test.score >= T0).astype(int)

print(f"true fraud rate   | thin-file {true_fraud[established==0].mean()*100:5.2f}%"
      f"   established {true_fraud[established==1].mean()*100:5.2f}%")
print(f"LABELLED fraud    | thin-file {observed_label[established==0].sum():5d}"
      f"       established {observed_label[established==1].sum():5d}")

true fraud rate   | thin-file  7.97%   established  5.55%
LABELLED fraud    | thin-file  1332       established   320


### The dashboard is green

In [3]:
print(f"AUC vs OBSERVED labels (the number they report) : {roc_auc_score(test.observed_label, test.score):.3f}")
print(f"AUC vs TRUE fraud      (the number nobody has)  : {roc_auc_score(test.true_fraud, test.score):.3f}\n")
for c, n in [(0, "thin-file  "), (1, "established")]:
    s = test[test.established == c]
    print(f"{n} | legit wrongly declined {s[s.true_fraud==0].d0.mean()*100:5.2f}%"
          f" | fraud caught {s[s.true_fraud==1].d0.mean()*100:5.2f}%")

AUC vs OBSERVED labels (the number they report) : 0.778
AUC vs TRUE fraud      (the number nobody has)  : 0.641

thin-file   | legit wrongly declined 15.94% | fraud caught 37.10%
established | legit wrongly declined  0.39% | fraud caught  1.79%


---
# ACT 1 — LEAK
No score. No gradients. No model access. **One bit back: approved, or declined.**
Behaviour is held fixed; only the *presentation* changes. This is card testing, repointed.

In [4]:
Q = [0]
def oracle(f, m, t, feats=FULL):
    """Production endpoint. Returns 1 = declined, 0 = approved. Nothing else."""
    Q[0] += len(f); return (m.predict_proba(f[feats])[:, 1] >= t).astype(int)

def probe(n, inst, ten, seed, email=None):
    r = np.random.default_rng(seed)
    return pd.DataFrame(dict(amount=np.exp(r.normal(4.6,.35,n)), hour=r.integers(0,24,n),
        tenure_days=np.full(n,ten,float), instrument=np.full(n,inst),
        bill_ship_match=np.zeros(n,int), device_reuse=r.poisson(1.2,n),
        velocity_1h=r.poisson(.7,n),
        email_age_days=r.gamma(2.0,200,n) if email is None else np.full(n,email,float)))

print("                tenure   20d     180d     900d")
for inst, nm in [(0,"prepaid"), (1,"debit  "), (2,"credit ")]:
    print(f"  {nm}              " + "".join(
        f"{oracle(probe(30,inst,t,900+inst*10+t), model, T0).mean()*100:7.1f}%" for t in (20,180,900)))
print(f"\n  probe transactions spent: {Q[0]}")

                tenure   20d     180d     900d
  prepaid                 80.0%   70.0%    0.0%
  debit                   70.0%   63.3%    6.7%
  credit                  73.3%   53.3%    0.0%

  probe transactions spent: 270


**The model just handed over its decision logic through a one-bit channel.** Tenure outranks instrument. The cliff sits between 180 and 900 days.

---
# ACT 2 — GAME
Same fraud. Same behaviour. One field changed.

In [5]:
fp = test[(test.true_fraud == 1) & (test.established == 0)].copy()
mg = fp.copy(); mg["instrument"] = 2; mg["tenure_days"] = 900.0
a0, m0 = oracle(fp, model, T0), oracle(mg, model, T0)

print(f"  fraud caught, as-is                 : {a0.mean()*100:5.2f}%")
print(f"  fraud caught, presented established : {m0.mean()*100:5.2f}%\n")
print(f"  ATTACKER CLEAR RATE : {(1-a0.mean())*100:.1f}%  ->  {(1-m0.mean())*100:.1f}%")
print(f"  MIGRATION ADVANTAGE : +{(a0.mean()-m0.mean())*100:.1f} percentage points")

  fraud caught, as-is                 : 37.10%
  fraud caught, presented established :  1.79%

  ATTACKER CLEAR RATE : 62.9%  ->  98.2%
  MIGRATION ADVANTAGE : +35.3 percentage points


No perturbation search. No gradient estimation. **One field.**

---
# ACT 3 — THE FIX EVERYONE SHIPS
Delete the offending features. Retrain. Re-run the attacker's exact script.

In [6]:
naive = fit(train, "observed_label", SAN)
sn = naive.predict_proba(test[SAN])[:, 1]
T1 = np.quantile(sn, .93); test["d1"] = (sn >= T1).astype(int)

a1, m1 = oracle(fp, naive, T1, SAN), oracle(mg, naive, T1, SAN)
print(f"  MIGRATION ADVANTAGE : +{(a0.mean()-m0.mean())*100:.1f}pp"
      f"  ->  +{(a1.mean()-m1.mean())*100:.1f}pp        <-- ticket closed?\n")
for c, n in [(0, "thin-file  "), (1, "established")]:
    s = test[test.established == c]
    print(f"  {n} | legit wrongly declined "
          f"{s[s.true_fraud==0].d0.mean()*100:5.2f}% -> {s[s.true_fraud==0].d1.mean()*100:5.2f}%")

  MIGRATION ADVANTAGE : +35.3pp  ->  +0.0pp        <-- ticket closed?

  thin-file   | legit wrongly declined 15.94% -> 14.29%
  established | legit wrongly declined  0.39% ->  1.33%


### The attacker spends one more afternoon

In [7]:
print("  re-probing on a feature you kept — email account age:\n")
for ea in (30, 400, 1200):
    print(f"    email age {ea:5d}d  ->  declined {oracle(probe(40,1,180,7,email=ea), naive, T1, SAN).mean()*100:6.1f}%")

  re-probing on a feature you kept — email account age:

    email age    30d  ->  declined  100.0%
    email age   400d  ->  declined   22.5%
    email age  1200d  ->  declined    0.0%


**You did not close the door. You moved it.**

The attack script returns zero, so your metric says fixed. The disparity barely
moved, and the leak reappeared on a feature nobody thought was sensitive.
You broke your own detection of the old lane and left the defect intact.

---
# ACT 4 — THE FIX THAT WORKS
Not an algorithm. A **random audit sample** — investigate 5% at random regardless
of cohort. Unbiased labels on a slice let you *measure* how badly the legacy
policy skewed the rest, and reweight accordingly.

In [8]:
AUDIT_FRAC, WEIGHT_CLIP = 0.05, 5.0
aud = rng.random(len(train)) < AUDIT_FRAC
train["cl"] = np.where(aud, train.true_fraud, train.observed_label)

prop = {c: max(train[aud & (train.established==c) & (train.true_fraud==1)].observed_label.mean(), .05)
        for c in (0, 1)}
print(f"  measured legacy label capture | thin-file {prop[0]*100:.0f}%   established {prop[1]*100:.0f}%\n")

w = np.where(train.cl == 1, np.minimum(1/train.established.map(prop).values, WEIGHT_CLIP), 1.0)
fixed = fit(train, "cl", FULL, w)
sf = fixed.predict_proba(test[FULL])[:, 1]
T2 = np.quantile(sf, .93); test["d2"] = (sf >= T2).astype(int)

a2, m2 = oracle(fp, fixed, T2), oracle(mg, fixed, T2)
print(f"  MIGRATION ADVANTAGE : +{(a0.mean()-m0.mean())*100:.1f}pp  ->  {(a2.mean()-m2.mean())*100:+.1f}pp\n")
for c, n in [(0, "thin-file  "), (1, "established")]:
    s = test[test.established == c]
    print(f"  {n} | legit declined {s[s.true_fraud==0].d0.mean()*100:5.2f}% -> {s[s.true_fraud==0].d2.mean()*100:5.2f}%"
          f" | fraud caught {s[s.true_fraud==1].d0.mean()*100:5.2f}% -> {s[s.true_fraud==1].d2.mean()*100:5.2f}%")

print("\n  re-probe email age — did the second door close too?\n")
for ea in (30, 400, 1200):
    print(f"    email age {ea:5d}d  ->  declined {oracle(probe(40,1,180,7,email=ea), fixed, T2).mean()*100:6.1f}%")

  measured legacy label capture | thin-file 81%   established 23%



  MIGRATION ADVANTAGE : +35.3pp  ->  -2.2pp

  thin-file   | legit declined 15.94% -> 10.20% | fraud caught 37.10% -> 30.82%
  established | legit declined  0.39% ->  3.55% | fraud caught  1.79% -> 12.01%

  re-probe email age — did the second door close too?

    email age    30d  ->  declined  100.0%
    email age   400d  ->  declined   50.0%
    email age  1200d  ->  declined   20.0%


### What it cost you

In [9]:
fr, lg = test[test.true_fraud == 1], test[test.true_fraud == 0]
print(f"  TOTAL fraud caught           : {fr.d0.mean()*100:5.2f}%  ->  {fr.d2.mean()*100:5.2f}%")
print(f"  TOTAL legit wrongly declined : {lg.d0.mean()*100:5.2f}%  ->  {lg.d2.mean()*100:5.2f}%")

  TOTAL fraud caught           : 18.57%  ->  20.95%
  TOTAL legit wrongly declined :  6.20%  ->   6.03%


---
### Same decline budget. More fraud caught. Fewer legitimate customers declined.
### The attacker's map is gone.

**You gave up nothing.**

---
*The email-age gradient is reduced, not eliminated — there is a genuine risk
differential in this population, so some signal should survive. This kills most
of the attack, not all of it. Anyone who tells you a fairness control zeroes an
attack surface is selling you something.*